In [15]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

#**Verificar se existem dados vazios**

In [3]:

# Importação do dataset.
url= "https://raw.githubusercontent.com/Fer412/Dataset_2026/refs/heads/main/heart1.csv"
df = pd.read_csv(url, sep=';')
df.to_csv('heart1.csv', index=False)
# Verificar valores nulos (corrigido)
print("\n=== VERIFICAÇÃO DE VALORES NULOS ===")
tem_nulos = df.isnull().any().any()
print(f"Existe algum valor nulo? {'Sim' if tem_nulos else 'Não'}")
print(f"Total de nulos por coluna:\n{df.isnull().sum()}")


=== VERIFICAÇÃO DE VALORES NULOS ===
Existe algum valor nulo? Não
Total de nulos por coluna:
age         0
sex         0
cp          0
trtbps      0
chol        0
fbs         0
restecg     0
thalachh    0
exng        0
oldpeak     0
slp         0
caa         0
thall       0
output      0
dtype: int64


In [4]:
print("=== INFORMAÇÕES DO DATASET ===")
print(f"Shape: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nPrimeiras 5 linhas:")
print(df.head())

=== INFORMAÇÕES DO DATASET ===
Shape: (303, 14)

Colunas: ['age', 'sex', 'cp', 'trtbps', 'chol', 'fbs', 'restecg', 'thalachh', 'exng', 'oldpeak', 'slp', 'caa', 'thall', 'output']

Primeiras 5 linhas:
   age sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  caa  \
0   63   M   3     145   233    1        0       150     0      2.3    0    0   
1   37   M   2     130   250    0        1       187     0      3.5    0    0   
2   41   F   1     130   204    0        0       172     0      1.4    2    0   
3   56   M   1     120   236    0        1       178     0      0.8    2    0   
4   57   F   0     120   354    0        1       163     1      0.6    2    0   

   thall  output  
0      1       1  
1      2       1  
2      2       1  
3      2       1  
4      2       1  


#**Verificar Dados Não Numéricos**

In [5]:
print(df.head(20))
print(df.dtypes)
# Tenta converter TUDO para numérico
# Valores não numéricos viram NaN
df_numerico = df.apply(pd.to_numeric, errors='coerce')

# Onde há NaN na conversão, originalmente não era numérico
problemas = df_numerico.isnull()

# Mostra posições com valores não numéricos
print("Posições com valores não numéricos:")
print(problemas)


    age sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  \
0    63   M   3     145   233    1        0       150     0      2.3    0   
1    37   M   2     130   250    0        1       187     0      3.5    0   
2    41   F   1     130   204    0        0       172     0      1.4    2   
3    56   M   1     120   236    0        1       178     0      0.8    2   
4    57   F   0     120   354    0        1       163     1      0.6    2   
5    57   M   0     140   192    0        1       148     0      0.4    1   
6    56   F   1     140   294    0        0       153     0      1.3    1   
7    44   M   1     120   263    0        1       173     0      0.0    2   
8    52   M   2     172   199    1        1       162     0      0.5    2   
9    57   M   2     150   168    0        1       174     0      1.6    2   
10   54   M   0     140   239    0        1       160     0      1.2    2   
11   48   F   2     130   275    0        1       139     0      0.2    2   

#**Codificação dos Valores Não numéricos**

In [6]:
label_encoder = LabelEncoder()

df['encoded_sex'] = label_encoder.fit_transform(df['sex'])
df['encoded_sex'] = df['encoded_sex'] + 1
df['sex'] = df['encoded_sex']

# Ver resultado
print(df[['sex', 'encoded_sex']].head())

   sex  encoded_sex
0    2            2
1    2            2
2    1            1
3    2            2
4    1            1


In [7]:
# Verificar a coluna 'sex' especificamente
print("\n=== ANÁLISE DA COLUNA 'sex' ===")
print(f"Valores únicos: {df['sex'].unique()}")
print(f"Contagem:\n{df['sex'].value_counts()}")


=== ANÁLISE DA COLUNA 'sex' ===
Valores únicos: [2 1]
Contagem:
sex
2    207
1     96
Name: count, dtype: int64


In [8]:
print(df.head())

   age  sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  \
0   63    2   3     145   233    1        0       150     0      2.3    0   
1   37    2   2     130   250    0        1       187     0      3.5    0   
2   41    1   1     130   204    0        0       172     0      1.4    2   
3   56    2   1     120   236    0        1       178     0      0.8    2   
4   57    1   0     120   354    0        1       163     1      0.6    2   

   caa  thall  output  encoded_sex  
0    0      1       1            2  
1    0      2       1            2  
2    0      2       1            1  
3    0      2       1            2  
4    0      2       1            1  


#**Gravar DataSet - heart_clean_1**

In [9]:
df.to_csv('heart_clean_1.csv', index=False)

#**Normalizar o dataset**

In [10]:

# Separar features (X) e target (y) - assumindo que a última coluna é o target
# Verificar qual coluna pode ser o target (geralmente 'target' ou 'condition')
colunas_target_candidatas = ['target', 'condition', 'num']
target_col = None
for col in colunas_target_candidatas:
    if col in df.columns:
        target_col = col
        break
if target_col is None:
    target_col = df.columns[-1]  # última coluna como fallback

print(f"\nColuna target identificada: '{target_col}'")

X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

# Normalização com StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Criar dataset normalizado (recombinar com target)
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)
df_scaled[target_col] = y.values

# Salvar heart_scale_2.csv
df_scaled.to_csv('heart_scale_2.csv', index=False)
print("\n✓ Dataset normalizado salvo como: heart_scale_2.csv")




Coluna target identificada: 'encoded_sex'
Features (X): (303, 14)
Target (y): (303,)

✓ Dataset normalizado salvo como: heart_scale_2.csv


#**Dataset de Treino e Teste**

In [11]:
# ============================================================================
# ATIVIDADE 3 - Criação de datasets de treino e teste
# ============================================================================
print("\n" + "="*60)
print("ATIVIDADE 3 - DIVISÃO TREINO/TESTE")
print("="*60)

# Divisão dos dados (70% treino, 30% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\nDataset de treino: {X_train.shape[0]} amostras")
print(f"Dataset de teste: {X_test.shape[0]} amostras")

# Criar DataFrames para salvar
df_treino = pd.DataFrame(X_train, columns=X.columns)
df_treino[target_col] = y_train.values

df_teste = pd.DataFrame(X_test, columns=X.columns)
df_teste[target_col] = y_test.values

# Salvar datasets
df_treino.to_csv('heart_treino_3.csv', index=False)
df_teste.to_csv('heart_teste_3.csv', index=False)


ATIVIDADE 3 - DIVISÃO TREINO/TESTE

Dataset de treino: 212 amostras
Dataset de teste: 91 amostras


#**MPLClassifier**

In [12]:

# ============================================================================
# ATIVIDADE 4 - Criação e treino da rede neural MLPClassifier
# ============================================================================
print("\n" + "="*60)
print("ATIVIDADE 4 - CRIAÇÃO E TREINO DA REDE NEURAL")
print("="*60)

# Configuração do MLPClassifier conforme especificado
mlp = MLPClassifier(
    hidden_layer_sizes=(150, 100, 50),
    max_iter=100000,
    activation='relu',
    solver='adam',
    random_state=42,
    verbose=True
)

print("\nArquitetura da rede neural:")
print(f"  - Camadas ocultas: 150, 100, 50")
print(f"  - Ativação: relu")
print(f"  - Otimizador: adam")
print(f"  - Iterações máximas: 100000")
print("\nIniciando treinamento...")

# Treinar o modelo
mlp.fit(X_train, y_train)

print("\n✓ Treinamento concluído!")
print(f"  - Iterações realizadas: {mlp.n_iter_}")
print(f"  - Perda final: {mlp.loss_:.4f}")



ATIVIDADE 4 - CRIAÇÃO E TREINO DA REDE NEURAL

Arquitetura da rede neural:
  - Camadas ocultas: 150, 100, 50
  - Ativação: relu
  - Otimizador: adam
  - Iterações máximas: 100000

Iniciando treinamento...
Iteration 1, loss = 0.84508138
Iteration 2, loss = 0.72350723
Iteration 3, loss = 0.63754871
Iteration 4, loss = 0.57674233
Iteration 5, loss = 0.52909462
Iteration 6, loss = 0.49087559
Iteration 7, loss = 0.45634516
Iteration 8, loss = 0.42390990
Iteration 9, loss = 0.39222349
Iteration 10, loss = 0.35958115
Iteration 11, loss = 0.32485115
Iteration 12, loss = 0.28914448
Iteration 13, loss = 0.25382006
Iteration 14, loss = 0.22012950
Iteration 15, loss = 0.18899290
Iteration 16, loss = 0.15985241
Iteration 17, loss = 0.13411041
Iteration 18, loss = 0.11105905
Iteration 19, loss = 0.09099393
Iteration 20, loss = 0.07396146
Iteration 21, loss = 0.05939890
Iteration 22, loss = 0.04738205
Iteration 23, loss = 0.03707944
Iteration 24, loss = 0.02919666
Iteration 25, loss = 0.02309373
Ite

#**Avaliação da exatidão da Rede Neural**

In [16]:

# ============================================================================
# ATIVIDADE 5 - Avaliação do modelo
# ============================================================================
print("\n" + "="*60)
print("ATIVIDADE 5 - AVALIAÇÃO DO MODELO")
print("="*60)

# Fazer predições
y_pred = mlp.predict(X_test)

# Calcular exatidão (accuracy)
acuracia = mlp.score(X_test, y_test)
print(f"\n=== EXATIDÃO (ACCURACY) ===")
print(f"Acurácia do modelo: {acuracia:.4f} ({acuracia*100:.2f}%)")

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
print(f"\n=== MATRIZ DE CONFUSÃO ===")
print(f"Formato da matriz: {cm.shape}")
print("\nMatriz de confusão:")
print("          Predito")
print("          Neg  Pos")
print(f"Real Neg  {cm[0,0]:3d}  {cm[0,1]:3d}")
print(f"     Pos  {cm[1,0]:3d}  {cm[1,1]:3d}")

# Métricas adicionais para melhor avaliação
if cm.shape == (2, 2):  # Classificação binária
    tn, fp, fn, tp = cm.ravel()
    sensibilidade = tp / (tp + fn) if (tp + fn) > 0 else 0
    especificidade = tn / (tn + fp) if (tn + fp) > 0 else 0
    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_score = 2 * (precisao * sensibilidade) / (precisao + sensibilidade) if (precisao + sensibilidade) > 0 else 0

    print(f"\n=== MÉTRICAS DETALHADAS ===")
    print(f"Sensibilidade (Recall): {sensibilidade:.4f}")
    print(f"Especificidade: {especificidade:.4f}")
    print(f"Precisão: {precisao:.4f}")
    print(f"F1-Score: {f1_score:.4f}")



ATIVIDADE 5 - AVALIAÇÃO DO MODELO

=== EXATIDÃO (ACCURACY) ===
Acurácia do modelo: 1.0000 (100.00%)

=== MATRIZ DE CONFUSÃO ===
Formato da matriz: (2, 2)

Matriz de confusão:
          Predito
          Neg  Pos
Real Neg   29    0
     Pos    0   62

=== MÉTRICAS DETALHADAS ===
Sensibilidade (Recall): 1.0000
Especificidade: 1.0000
Precisão: 1.0000
F1-Score: 1.0000


#**Script de predição**

In [17]:

# ============================================================================
# ATIVIDADE 6 - Script para predição usando o modelo criado
# ============================================================================
print("\n" + "="*60)
print("ATIVIDADE 6 - SCRIPT DE PREDIÇÃO")
print("="*60)

def fazer_predicao(modelo, scaler, novos_dados, colunas_features, target_original=None):
    """
    Função para fazer predições com o modelo treinado.

    Parâmetros:
    - modelo: MLPClassifier treinado
    - scaler: StandardScaler ajustado
    - novos_dados: dict ou DataFrame com novos dados
    - colunas_features: lista com nomes das colunas features
    - target_original: LabelEncoder original (se houver codificação)

    Retorna:
    - Predição(ões)
    """
    # Converter para DataFrame se for dicionário
    if isinstance(novos_dados, dict):
        df_novo = pd.DataFrame([novos_dados])
    elif isinstance(novos_dados, pd.DataFrame):
        df_novo = novos_dados.copy()
    else:
        raise ValueError("novos_dados deve ser dict ou DataFrame")

    # Garantir ordem correta das colunas
    df_novo = df_novo[colunas_features]

    # Aplicar normalização
    X_novo_scaled = scaler.transform(df_novo)

    # Fazer predição
    predicao = modelo.predict(X_novo_scaled)

    return predicao

# Exemplo de uso da função de predição
print("\n=== EXEMPLO DE PREDIÇÃO ===")

# Criar um exemplo de novo paciente (usando valores médios normalizados)
# Nota: Estes valores devem corresponder às features normalizadas
colunas_features = X.columns.tolist()
print(f"Features esperadas: {colunas_features}")

# Exemplo com valores médios (0 na escala normalizada)
exemplo_paciente = {col: 0.0 for col in colunas_features}

print(f"\nDados do novo paciente (valores normalizados):")
for col, valor in exemplo_paciente.items():
    print(f"  {col}: {valor}")

# Fazer predição
try:
    pred = fazer_predicao(mlp, scaler, exemplo_paciente, colunas_features)
    print(f"\n✅ PREDIÇÃO: {'DOENÇA CARDÍACA POSITIVA' if pred[0] == 1 else 'DOENÇA CARDÍACA NEGATIVA'}")
    print(f"   (Classe predita: {pred[0]})")
except Exception as e:
    print(f"Erro na predição: {e}")
    print("Ajuste o exemplo conforme as colunas reais do dataset.")

# Função para predição interativa
print("\n=== PREDIÇÃO INTERATIVA ===")
print("Para usar a função de predição com novos dados:")
print("""
# Exemplo de uso:
novo_paciente = {
    'age': 0.5,      # valor normalizado
    'sex': -0.2,     # valor normalizado
    'cp': 1.0,       # valor normalizado
    # ... outras features normalizadas
}

predicao = fazer_predicao(mlp, scaler, novo_paciente, colunas_features)
print(f"Resultado: {predicao[0]}")
""")

# ============================================================================
# RESUMO FINAL
# ============================================================================
print("\n" + "="*60)
print("RESUMO FINAL DO LABORATÓRIO 6")
print("="*60)
print("""
Arquivos gerados:
  1. heart_clean_1.csv    - Dataset limpo (sem missing values, com codificação)
  2. heart_scale_2.csv    - Dataset normalizado
  3. heart_treino_3.csv   - Dataset de treino (70%)
  4. heart_teste_3.csv    - Dataset de teste (30%)

Modelo treinado:
  - MLPClassifier com arquitetura: (150, 100, 50)
  - Activation: relu | Solver: adam | max_iter: 100000

Métricas de avaliação:
  - Accuracy: {:.4f} ({:.2f}%)
  - Matriz de confusão gerada

Predição:
  - Função 'fazer_predicao()' disponível para classificar novos dados
""".format(acuracia, acuracia*100))




ATIVIDADE 6 - SCRIPT DE PREDIÇÃO

=== EXEMPLO DE PREDIÇÃO ===
Features esperadas: ['age', 'sex', 'cp', 'trtbps', 'chol', 'fbs', 'restecg', 'thalachh', 'exng', 'oldpeak', 'slp', 'caa', 'thall', 'output']

Dados do novo paciente (valores normalizados):
  age: 0.0
  sex: 0.0
  cp: 0.0
  trtbps: 0.0
  chol: 0.0
  fbs: 0.0
  restecg: 0.0
  thalachh: 0.0
  exng: 0.0
  oldpeak: 0.0
  slp: 0.0
  caa: 0.0
  thall: 0.0
  output: 0.0

✅ PREDIÇÃO: DOENÇA CARDÍACA POSITIVA
   (Classe predita: 1)

=== PREDIÇÃO INTERATIVA ===
Para usar a função de predição com novos dados:

# Exemplo de uso:
novo_paciente = {
    'age': 0.5,      # valor normalizado
    'sex': -0.2,     # valor normalizado
    'cp': 1.0,       # valor normalizado
    # ... outras features normalizadas
}

predicao = fazer_predicao(mlp, scaler, novo_paciente, colunas_features)
print(f"Resultado: {predicao[0]}")


RESUMO FINAL DO LABORATÓRIO 6

Arquivos gerados:
  1. heart_clean_1.csv    - Dataset limpo (sem missing values, com codific